# TP7 : ajouter un broker, réassigner, redémarrer sans interruption, décommissionner

| Repère | Valeur |
|---|---|
| Moment | Jour 2, 13:40-13:55 : étapes 6.1 et 6.3 jouées par le formateur (démo exploitation) ; TP complet en autonomie après la formation |
| Compétences | C14 Réassigner des partitions, ajouter et retirer un broker, ajouter un log dir ; C15 Rolling restart sans interruption de service |
| Slides de rappel | S61 (log dirs), S64 (configuration statique et dynamique), S65 (réassignation), S66 (ajout), S67 (décommissionnement), S68 (rolling restart) |
| Version Markdown | `enonce.md`, mêmes étapes en ligne de commande seule |

**Mode d'emploi du notebook**

- Exécutez les cellules dans l'ordre avec `Maj+Entrée`. Après un redémarrage du noyau, les variables sont perdues : relancez la cellule d'outils, puis les cellules qui définissent les fonctions et les variables dont vous avez besoin. Ne relancez pas une cellule qui modifie le cluster (création d'un topic, arrêt d'un nœud) : elle échouerait, ou referait l'action.
- Chaque étape suit le même schéma : une explication, un **exemple en ligne de commande**, une **zone `# TODO`**, puis une **vérification**. Les commandes qui modifient le cluster sont affichées sans être exécutées (« ne pas lancer en plus du TODO »).
- Le client Python n'a pas d'API pour la réassignation, les répertoires de données (log dirs), le quorum ni le désenregistrement d'un broker. Dans ce TP, votre code Python **pilote les outils Java** par des cellules `!docker exec kafka-tools ...` et **exploite leurs sorties** : les plans de réassignation et l'état des log dirs sont en JSON.
- Les plans sont écrits dans `/tmp` du conteneur `tools`, comme dans la version Markdown.

In [ ]:
# Connexion au lab : adresses des brokers, configuration du client Python, dossier de travail
import os
import time

from confluent_kafka.admin import AdminClient

BS = os.environ["BS"]  # listener INTERNAL, comme $BS dans le conteneur tools
CONF = {"bootstrap.servers": BS, "log_level": 2}

# Les commandes docker compose et les fichiers config/ se lisent depuis le dossier lab/
LAB_DIR = os.environ["FORMATION_DIR"] + "/lab"
os.chdir(LAB_DIR)
os.environ["COMPOSE_PROGRESS"] = "plain"  # docker compose sans animation dans le notebook

In [ ]:
# Deux fonctions d'aide, écrites ici : elles ne font pas partie de confluent-kafka
def wait_for(condition, timeout=60, step=2, message="condition"):
    """Rappelle condition() jusqu'à obtenir une valeur vraie, et la renvoie."""
    start = time.time()
    while True:
        try:
            value = condition()
        except Exception:
            value = None
        if value:
            return value
        if time.time() - start >= timeout:
            raise TimeoutError("%s : non atteinte en %d s" % (message, timeout))
        time.sleep(step)

def table(lines, headers):
    """Affiche des lignes en tableau Markdown dans le notebook."""
    from IPython.display import Markdown, display

    md = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
    md += ["| " + " | ".join(str(v).replace("|", "\\|") for v in line) + " |" for line in lines]
    display(Markdown("\n".join(md)))

In [ ]:
import json
import os
import shutil
import threading
import time
from collections import Counter

from confluent_kafka import ElectionType, KafkaException, Producer
from confluent_kafka.admin import AlterConfigOpType, ConfigEntry, ConfigResource, ConfigSource, NewTopic


def placement(topics=None):
    """{(topic, partition): (leader, [replicas])}, pour les topics donnés ou pour tous les topics."""
    admin = AdminClient(CONF)
    meta = admin.list_topics(timeout=10).topics
    return {(t, p.id): (p.leader, p.replicas) for t in (topics or meta) for p in meta[t].partitions.values()}


def replicas_per_broker(topics=None):
    """Nombre de répliques hébergées par chaque broker."""
    return dict(sorted(Counter(b for _, replicas in placement(topics).values() for b in replicas).items()))


def urp():
    """Nombre de partitions sous-répliquées."""
    admin = AdminClient(CONF)
    return sum(1 for t in admin.list_topics(timeout=10).topics.values()
               for p in t.partitions.values() if len(p.isrs) < len(p.replicas))


def log_dirs(broker, topics=None):
    """{repertoire: {partition: taille}} d'un broker, lu dans la sortie JSON de kafka-log-dirs.sh."""
    options = " --topic-list " + ",".join(topics) if topics else ""
    output = !docker exec kafka-tools bash -c "kafka-log-dirs.sh --bootstrap-server $BS --describe --broker-list {broker}{options} | tail -1" 2>&1
    output = output.n
    return {d["logDir"]: {p["partition"]: p["size"] for p in d["partitions"]}
            for b in json.loads(output)["brokers"] for d in b["logDirs"]}


def healthy(node):
    output = !docker compose ps --format '{{{{.Health}}}}' kafka-{node} 2>&1
    return output.n == "healthy"


admin = AdminClient(CONF)
print("Brokers :", sorted(n.id for n in admin.describe_cluster().result().nodes), "- URP :", urp())

## 1. Objectifs

À la fin du TP, vous savez :

- ajouter un broker à un cluster KRaft et vérifier son enregistrement ;
- produire, exécuter et suivre un plan de réassignation, avec une limite de débit ;
- rééquilibrer les leaders après une réassignation ;
- ajouter un répertoire de données à un nœud et y déplacer une réplique ;
- réaliser un rolling restart sous charge, sans perte ni échec côté producteur ;
- retirer un broker proprement : cordon, réassignation, arrêt, désenregistrement.

## 2. Prérequis

- Lab en mode `extended`, sain : `scripts/validate.sh`. Lab démarré par `scripts/start.sh` (sinon `docker compose` ne peut pas créer `kafka-4`).
- Topics `orders` (6 partitions) et `payments` (3 partitions) présents, issus des TP2 et TP4.
- Aucune partition sous-répliquée.
- Notions : slides S61 à S68.

## 3. Architecture utilisée

```
  Avant                                  Pendant le TP                         Après 6.7
  kafka-1  broker+contrôleur  az-a       kafka-1  data + data2 (6.5)            kafka-1  data + data2
  kafka-2  broker+contrôleur  az-b       kafka-2                                kafka-2
  kafka-3  broker+contrôleur  az-c       kafka-3                                kafka-3
                                         kafka-4  broker seul, az-a             (retiré, désenregistré)
                                                  observateur du quorum
```

| Nœud | Rôles | Rack | Remarque |
|---|---|---|---|
| kafka-1 à kafka-3 | broker, controller (votants) | az-a, az-b, az-c | Quorum de 3 votants |
| kafka-4 | broker seul | az-a | Profil Compose `scale` ; formaté avec `--no-initial-controllers` |

| Lab | Production |
|---|---|
| Nœuds combinés broker et contrôleur | Contrôleurs dédiés (3 ou 5), sur leurs propres machines |
| Le nouveau répertoire est formaté par le script de démarrage (`format --ignore-formatted`) | Formatage du disque ajouté : étape manuelle de la procédure, avant le redémarrage |
| Plans générés par `--generate`, qui ne tient compte que du nombre de répliques | Outil de rééquilibrage qui tient compte de la charge (Cruise Control, par exemple) |

## 4. Contexte métier

Le Black Friday approche. NovaShop ajoute un quatrième broker et doit y répartir les topics critiques sans interrompre les ventes. Le disque de `kafka-1` est presque plein : un second disque est installé. Un correctif de sécurité impose ensuite un redémarrage de tous les nœuds en pleine journée. Enfin, après l'événement, le broker temporaire est rendu.

## 5. Énoncé

1. Démarrez `kafka-4` et prouvez qu'il fait partie du cluster.
2. Montrez qu'il ne reçoit aucune partition existante.
3. Répartissez `orders-history`, puis `orders` et `payments`, sur les 4 brokers, avec une limite de débit.
4. Ajoutez le répertoire `/var/lib/kafka/data2` à `kafka-1` et déplacez-y une réplique de `payments`.
5. Redémarrez les 3 nœuds d'origine un par un, sous un producteur `acks=all`, sans aucun envoi en échec.
6. Retirez `kafka-4` du cluster sans perdre de réplique.

## 6. Étapes

### 6.1 Démarrer kafka-4

**Exemple en ligne de commande** : les paramètres clés du nouveau nœud.

In [ ]:
!grep -E '^(process.roles|node.id|controller.quorum.bootstrap.servers|broker.rack|log.dirs)=' config/kafka/kafka-4/server.properties

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [hôte]
docker compose --profile scale up -d kafka-4
```

In [ ]:
# TODO 6.1 : démarrer kafka-4 et prouver qu'il est enregistré
# 1. Démarrez le nœud : `!docker compose --profile scale up -d kafka-4`.
# 2. Attendez (wait_for, timeout=90) que le broker 4 figure dans admin.describe_cluster().result().nodes.
# 3. brokers : {identifiant: rack} des nœuds de describe_cluster() (attributs id et rack).
# 4. startup_logs : lignes de `!docker compose logs kafka-4` qui contiennent start-node, Formatting
#    ou Kafka Server started (grep -E).
brokers = {}
startup_logs = ""

print(brokers)

In [ ]:
# Vérification 6.1
assert brokers == {1: "az-a", 2: "az-b", 3: "az-c", 4: "az-a"}, "À revoir : 4 brokers, kafka-4 dans le rack az-a"
print("OK  4 brokers, kafka-4 dans le rack az-a")
assert "Formatting metadata directory" in startup_logs and "Kafka Server started" in startup_logs, "À revoir : kafka-4 formaté puis démarré"
print("OK  kafka-4 formaté puis démarré")

Statut dans le quorum et contenu du répertoire de données de `kafka-4` (lecture seule) :

In [ ]:
!docker exec kafka-tools kafka-metadata-quorum.sh --bootstrap-server $BS describe --replication
!docker exec kafka-tools kafka-log-dirs.sh --bootstrap-server $BS --describe --broker-list 4 | tail -1

**Résultat attendu** :

- `process.roles=broker`, `node.id=4`, `controller.quorum.bootstrap.servers` identique aux autres nœuds, `broker.rack=az-a` ;
- traces : `[start-node] Formatage du stockage (cluster W5FmqobtRp-BMcXlZhVKBw)`, `Formatting metadata directory /var/lib/kafka/data with metadata.version 4.3-IV0.`, puis `Kafka Server started` ;
- 4 brokers ; dans le quorum, 3 lignes `Leader` ou `Follower`, et une ligne `4 ... Observer` avec un lag de 0 ;
- répertoire de `kafka-4` vide : `"logDir":"/var/lib/kafka/data"` avec `"partitions":[]`.

### 6.2 Constater la répartition

Les fonctions `placement()` et `replicas_per_broker()` de la cellule d'outils lisent le placement des répliques.

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-topics.sh --bootstrap-server $BS --create --topic orders-history --partitions 4 --replication-factor 3
kafka-producer-perf-test.sh --bootstrap-server $BS --topic orders-history \
  --num-records 20000 --record-size 500 --throughput -1
```

In [ ]:
# TODO 6.2 : répartition avant et après un nouveau topic
# 1. before : replicas_per_broker(["orders", "payments"])
# 2. Créez orders-history (4 partitions, RF 3) et écrivez-y 20000 messages de 500 octets avec un Producer
#    (b"x" * 500 ; poll(0) après chaque produce, et en cas de BufferError, poll(0.1) puis nouvel essai ; flush() à la fin).
# 3. history : replicas_per_broker(["orders-history"])
before = history = {}

print("orders et payments :", before, "- orders-history :", history)

In [ ]:
# Vérification 6.2
assert 4 not in before, "À revoir : aucune réplique existante sur le broker 4"
print("OK  aucune réplique existante sur le broker 4")
assert history.get(4, 0) > 0, "À revoir : le nouveau topic utilise le broker 4 : %s" % history
print("OK  le nouveau topic utilise le broker 4 : %s" % history)

**Résultat attendu** : aucune réplique d'`orders` ni de `payments` sur le broker 4 ; `orders-history` en a plusieurs. Un nouveau topic utilise tous les brokers disponibles, les anciens ne bougent pas.

### 6.3 Premier cycle de réassignation

`kafka-reassign-partitions.sh --generate` propose un plan à partir d'une liste de topics et de brokers. Sa sortie contient deux lignes JSON, chacune après un titre : `Current partition replica assignment` (la répartition actuelle, votre retour arrière) et `Proposed partition reassignment configuration` (le plan proposé).

Les deux fonctions ci-dessous sont fournies : `execute(plan, rate)` écrit le plan dans `/tmp/plan.json` du conteneur `tools` et lance `--execute` (avec `--throttle` si un débit est donné) ; `verify_plan()` lance `--verify` sur ce fichier.

**Exemple en ligne de commande** : un plan généré pour `orders-history` (lecture seule : `--generate` ne change rien).

In [ ]:
def execute(plan, rate=None):
    !echo '{json.dumps(plan)}' | docker exec -i kafka-tools tee /tmp/plan.json > /dev/null
    option = " --throttle %d" % rate if rate else ""
    output = !docker exec kafka-tools bash -c "kafka-reassign-partitions.sh --bootstrap-server $BS --reassignment-json-file /tmp/plan.json --execute{option}" 2>&1
    return output.n


def verify_plan():
    output = !docker exec kafka-tools kafka-reassign-partitions.sh --bootstrap-server $BS --reassignment-json-file /tmp/plan.json --verify 2>&1
    return output.n


!echo '{{"version":1,"topics":[{{"topic":"orders-history"}}]}}' | docker exec -i kafka-tools tee /tmp/topics.json
!docker exec kafka-tools bash -c "kafka-reassign-partitions.sh --bootstrap-server $BS --topics-to-move-json-file /tmp/topics.json --broker-list 1,2,3,4 --generate | cut -c1-200"

In [ ]:
# TODO 6.3 a : les fonctions generate() et broker_throttles(), puis un premier plan limité
# 1. generate(topics, brokers) : écrit {"version": 1, "topics": [{"topic": t}, ...]} dans /tmp/topics.json du
#    conteneur tools (`!echo '...' | docker exec -i kafka-tools tee /tmp/topics.json`, avec json.dumps), lance --generate avec
#    --broker-list construit à partir de brokers ("1,2,3,4"), puis renvoie (current, proposed) : les deux lignes JSON
#    qui suivent les titres, lues avec json.loads (lines = output.splitlines() ; lines.index(titre) + 1).
# 2. broker_throttles(broker) : {name: value} des entrées de describe_configs sur ConfigResource("broker", str(broker))
#    dont le nom contient "throttled" et dont la source est ConfigSource.DYNAMIC_BROKER_CONFIG.value.
# 3. rollback, plan = generate(["orders-history"], [1, 2, 3, 4]) ; execute(plan, 50000000) ;
#    aussitôt après, throttles_4 = broker_throttles(4).
def generate(topics, brokers):
    return {}, {}


def broker_throttles(broker):
    return {}


rollback = plan = {}
throttles_4 = {}

print("partitions du plan :", len(plan.get("partitions", [])), "- limitations sur le broker 4 :", throttles_4)

In [ ]:
# Vérification 6.3 a
assert len(plan.get("partitions", [])) == 4 and len(rollback.get("partitions", [])) == 4, "À revoir : plan et retour arrière pour les 4 partitions"
print("OK  plan et retour arrière pour les 4 partitions")
assert throttles_4.get("leader.replication.throttled.rate") == "50000000", "À revoir : limitation posée sur le broker 4"
print("OK  limitation posée sur le broker 4")

Équivalent en ligne de commande du `TODO` b (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-reassign-partitions.sh --bootstrap-server $BS --reassignment-json-file /tmp/plan.json --verify
kafka-leader-election.sh --bootstrap-server $BS --election-type preferred --all-topic-partitions
```

In [ ]:
# TODO 6.3 b : suivre jusqu'à la fin, puis réélire les leaders
# 1. wait_until_done(timeout) : relance verify_plan() toutes les 5 s (wait_for, step=5) jusqu'à ce que sa sortie
#    contienne "Clearing topic-level throttles", et renvoie cette dernière sortie (wait_for() renvoie la valeur
#    vraie obtenue : faites renvoyer None à la condition tant que ce n'est pas fini).
# 2. end_73 = wait_until_done(120) ; throttles_after = broker_throttles(4).
# 3. non_preferred : nombre de partitions d'orders-history dont le leader n'est pas le premier de la liste des répliques.
# 4. Lancez l'élection des leaders préférés (elect_leaders, comme au TP4), puis recalculez non_preferred_after.
def wait_until_done(timeout):
    return ""


end_73 = ""
throttles_after = None
non_preferred = non_preferred_after = None

print("leaders non préférés avant l'élection :", non_preferred, "- après :", non_preferred_after)

In [ ]:
# Vérification 6.3 b
assert "Clearing broker-level throttles" in end_73, "À revoir : --verify a retiré les limitations"
print("OK  --verify a retiré les limitations")
assert throttles_after == {}, "À revoir : plus aucune limitation sur le broker 4"
print("OK  plus aucune limitation sur le broker 4")
assert non_preferred_after == 0, "À revoir : chaque partition d'orders-history a son leader préféré"
print("OK  chaque partition d'orders-history a son leader préféré")

**Résultat attendu** :

- le plan proposé utilise `"log_dirs":["any","any","any"]` ; `--execute` répond `The inter-broker throttle limit was set to 50000000 B/s` ;
- juste après : `leader.replication.throttled.rate` et `follower.replication.throttled.rate` à 50000000 sur le broker 4 ;
- `--verify` : `Reassignment of partition orders-history-N is completed.` pour les 4 partitions, puis `Clearing broker-level throttles on brokers 1,2,3,4` et `Clearing topic-level throttles on topic orders-history` ;
- après réassignation, certains leaders ne sont plus le premier de leur liste ; l'élection les remet en tête.

### 6.4 Étendre `orders` et `payments` à kafka-4

Même cycle, avec vos fonctions, et une limite de 1 Mo/s. La fonction `log_dirs(broker, topics)` de la cellule d'outils lit la taille des répliques d'un broker.

In [ ]:
# TODO 6.4 : orders et payments sur les 4 brokers
# 1. rollback_74, plan_74 = generate(["orders", "payments"], [1, 2, 3, 4]) ; execute(plan_74, 1000000).
# 2. first_verify : la sortie de verify_plan() aussitôt après.
# 3. Attendez la fin : wait_until_done(180).
# 4. on_4 : log_dirs(4, ["orders", "payments"]) ; distribution_74 : replicas_per_broker(["orders", "payments"]).
first_verify = ""
on_4 = distribution_74 = {}

print("sur le broker 4 :", on_4)
print("répartition :", distribution_74)

In [ ]:
# Vérification 6.4
assert "Status of partition reassignment" in first_verify, "À revoir : suivi lancé aussitôt après --execute"
print("OK  suivi lancé aussitôt après --execute")
assert distribution_74.get(4, 0) > 0 and broker_throttles(4) == {}, "À revoir : répliques sur le broker 4, limitations retirées"
print("OK  répliques sur le broker 4, limitations retirées")

**Résultat attendu** : `The inter-broker throttle limit was set to 1000000 B/s`. Selon le volume des partitions, le premier `--verify` montre des partitions `still in progress` ; avec des partitions très petites, tout peut être `completed` dès le premier `--verify`. Le broker 4 héberge ensuite des répliques des deux topics, avec leur taille en octets.

### 6.5 Défi : ajouter un log dir

Le paramètre `log.dirs` de `config/kafka/kafka-1/server.properties` liste les répertoires de données du nœud, séparés par des virgules. Le fichier est lisible et modifiable depuis le notebook (`KAFKA_1_FILE`). C'est une configuration **statique** : le nœud doit redémarrer pour la prendre en compte.

Un plan de réassignation peut aussi déplacer une réplique **entre les répertoires d'un même broker** : `log_dirs` contient un élément par réplique, dans le même ordre que `replicas`, et la valeur `"any"` laisse le broker choisir.

```json
{"version":1,"partitions":[{"topic":"payments","partition":<n>,"replicas":[<répliques actuelles, dans l'ordre>],"log_dirs":[<un élément par réplique>]}]}
```

In [ ]:
KAFKA_1_FILE = os.path.join(LAB_DIR, "config/kafka/kafka-1/server.properties")
!grep '^log.dirs' config/kafka/kafka-1/server.properties
print(log_dirs(1, ["payments"]))

Équivalent en ligne de commande du `TODO` a (**ne pas lancer en plus du TODO**) :

```bash
# [hôte]
sed -i.bak 's|^log.dirs=/var/lib/kafka/data$|log.dirs=/var/lib/kafka/data,/var/lib/kafka/data2|' \
  config/kafka/kafka-1/server.properties
docker compose restart kafka-1
```

In [ ]:
# TODO 6.5 a : ajouter /var/lib/kafka/data2 à kafka-1
# 1. Copiez KAFKA_1_FILE en KAFKA_1_FILE + ".bak" (shutil.copyfile), puis remplacez dans le fichier la ligne
#    log.dirs=/var/lib/kafka/data par log.dirs=/var/lib/kafka/data,/var/lib/kafka/data2.
# 2. Redémarrez kafka-1 (`!docker compose restart kafka-1`), puis attendez qu'il soit sain (healthy(1)) et que urp() vaille 0.
# 3. format_logs : lignes des traces récentes de kafka-1 qui contiennent start-node ou Formatting
#    (docker compose logs kafka-1 --since 2m).
# 4. log_dirs_1 : liste triée des répertoires de log_dirs(1).
format_logs = ""
log_dirs_1 = []

print(log_dirs_1)

In [ ]:
# Vérification 6.5 a
assert "Formatting data directory /var/lib/kafka/data2" in format_logs, "À revoir : data2 formaté au démarrage"
print("OK  data2 formaté au démarrage")
assert log_dirs_1 == ["/var/lib/kafka/data", "/var/lib/kafka/data2"], "À revoir : kafka-1 a deux répertoires"
print("OK  kafka-1 a deux répertoires")

In [ ]:
# TODO 6.5 b (défi) : déplacer une seule réplique de payments vers data2
# 1. Choisissez une partition de payments dont kafka-1 héberge une réplique : placement(["payments"]).
# 2. plan_logdir : le plan d'une seule partition, avec les répliques actuelles dans l'ordre et "log_dirs" qui
#    place la réplique du broker 1 dans /var/lib/kafka/data2 ("any" pour les autres).
# 3. execute(plan_logdir), puis attendez que verify_plan() contienne "completed successfully".
# 4. after_logdir : log_dirs(1, ["payments"]).
plan_logdir = {}
after_logdir = {}

print(json.dumps(plan_logdir))
print(after_logdir)

In [ ]:
# Vérification 6.5 b
moved = list(after_logdir.get("/var/lib/kafka/data2", {}))
assert len(moved) == 1 and moved[0].startswith("payments-"), "À revoir : une réplique de payments dans data2 : %s" % moved
print("OK  une réplique de payments dans data2 : %s" % moved)
assert not set(moved) & set(after_logdir.get("/var/lib/kafka/data", {})), "À revoir : elle n'est plus dans data"
print("OK  elle n'est plus dans data")

In [ ]:
!docker compose exec -T kafka-1 ls /var/lib/kafka/data2 </dev/null

**Résultat attendu** :

- traces : `Formatting data directory /var/lib/kafka/data2 with metadata.version 4.3-IV0.` ; le répertoire d'origine n'est pas reformaté ;
- `--execute` : `Successfully started moving log directory to /var/lib/kafka/data2 for replica payments-N with broker 1` ;
- `--verify` : `Reassignment of replica payments-N-1 completed successfully.` ;
- la partition apparaît sous `/var/lib/kafka/data2`, et plus sous `/var/lib/kafka/data`.

### 6.6 Rolling restart sous charge

La classe `Load` du TP4 revient : un producteur en arrière-plan, ici `acks=all` et idempotent. Pendant qu'il écrit 90 000 messages à 300 messages/s (5 minutes), redémarrez les nœuds 1 à 3 **un par un**, en attendant entre deux nœuds qu'il soit sain et que l'URP revienne à 0.

In [ ]:
class Load:
    """Producteur de charge en arrière-plan : count messages, à rate messages par seconde."""

    def __init__(self, topic, count, rate, config):
        self.confirmed = 0
        self.errors = []
        self.producer = Producer({"bootstrap.servers": BS, "log_level": 4, **config})
        self._thread = threading.Thread(target=self._send, args=(topic, count, rate), daemon=True)
        self._thread.start()

    def _report(self, err, msg):
        if err is None:
            self.confirmed += 1
        else:
            self.errors.append(err)

    def _send(self, topic, count, rate):
        start = time.time()
        for i in range(count):
            pause = start + i / rate - time.time()
            if pause > 0:
                time.sleep(pause)
            self.producer.produce(topic, value=b"x" * 200, on_delivery=self._report)
            self.producer.poll(0)
        self.producer.flush(120)

    def running(self):
        return self._thread.is_alive()

    def __repr__(self):
        return "Charge(%d confirmés, %d erreurs%s)" % (self.confirmed, len(self.errors),
                                                      ", en cours" if self.running() else ", terminée")

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**), pour chaque nœud `N` :

```bash
# [hôte]
docker compose restart kafka-N
docker compose ps kafka-N        # jusqu'à (healthy)
# [tools]
kafka-topics.sh --bootstrap-server $BS --describe --under-replicated-partitions | wc -l   # jusqu'à 0
```

**Exécutez la cellule `TODO` une seule fois** : elle dure environ 5 minutes.

In [ ]:
# TODO 6.6 : rolling restart sous charge
# 1. quorum_leader : le contrôleur actif, entier lu après "LeaderId:" dans
#    `output = !docker exec kafka-tools kafka-metadata-quorum.sh --bootstrap-server $BS describe --status` (output.n joint les lignes).
#    order : liste des nœuds 1, 2, 3, avec quorum_leader en dernier.
# 2. load = Load("payments", 90000, 300, {"acks": "all", "enable.idempotence": True})
# 3. Pour chaque nœud de order : `!docker compose restart kafka-{n}`, puis attendez healthy(n) et urp() == 0
#    (timeout=120) ; ajoutez à restart_log le tuple (nœud, secondes depuis le début, load.confirmed).
# 4. Attendez la fin de la charge (wait_for(lambda: not load.running(), timeout=360, step=5)).
# 5. Relancez l'élection des leaders préférés.
quorum_leader = None
order = []
restart_log = []
load = None

table(restart_log, ["Nœud", "Sain, URP à 0 (s)", "Messages confirmés"])
print(load)

In [ ]:
# Vérification 6.6
assert sorted(order) == [1, 2, 3] and order[-1] == quorum_leader and len(restart_log) == 3, "À revoir : 3 nœuds redémarrés, contrôleur actif (%s) en dernier : %s" % (quorum_leader, order)
print("OK  3 nœuds redémarrés, contrôleur actif (%s) en dernier : %s" % (quorum_leader, order))
assert load is not None and load.confirmed == 90000 and not load.errors, "À revoir : 90000 messages confirmés, aucune erreur : %s" % load
print("OK  90000 messages confirmés, aucune erreur : %s" % load)
assert urp() == 0, "À revoir : aucune partition sous-répliquée"
print("OK  aucune partition sous-répliquée")

**Résultat attendu** :

- le contrôleur actif est redémarré en dernier ;
- chaque nœud redevient sain en 10 à 20 s, et l'URP revient à 0 en quelques secondes ;
- 90 000 messages confirmés, aucune erreur définitive : le producteur a suivi les changements de leader.

### 6.7 Décommissionner kafka-4

Le **cordon** (`cordoned.log.dirs=*`, configuration dynamique du broker) empêche toute nouvelle réplique d'arriver sur le broker. Ensuite : réassigner ses répliques vers les autres brokers, l'arrêter, puis le désenregistrer.

Équivalent en ligne de commande du `TODO` a (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-configs.sh --bootstrap-server $BS --alter --entity-type brokers --entity-name 4 \
  --add-config 'cordoned.log.dirs=*'
```

In [ ]:
# TODO 6.7 a : cordon et réassignation vers 1, 2, 3
# 1. Cordonnez le broker 4 en Python : incremental_alter_configs sur ConfigResource("broker", "4") avec
#    ConfigEntry("cordoned.log.dirs", "*", incremental_operation=AlterConfigOpType.SET) (comme au TP4).
# 2. topics_on_4 : liste triée des topics qui ont une réplique sur le broker 4 (placement()).
# 3. rollback_77, plan_77 = generate(topics_on_4, [1, 2, 3]) ; plan_brokers : ensemble des brokers cités dans plan_77.
# 4. execute(plan_77, 50000000), puis wait_until_done(180). left_on_4 : log_dirs(4).
topics_on_4 = []
plan_brokers = set()
left_on_4 = None

print(topics_on_4, plan_brokers, left_on_4)

In [ ]:
# Vérification 6.7 a
assert plan_brokers == {1, 2, 3}, "À revoir : le plan n'utilise que les brokers 1, 2 et 3"
print("OK  le plan n'utilise que les brokers 1, 2 et 3")
assert left_on_4 == {"/var/lib/kafka/data": {}}, "À revoir : le broker 4 n'héberge plus aucune réplique"
print("OK  le broker 4 n'héberge plus aucune réplique")

Équivalent en ligne de commande du `TODO` b (**ne pas lancer en plus du TODO**) :

```bash
# [hôte]
docker compose --profile scale stop kafka-4
# [tools]
kafka-cluster.sh unregister --bootstrap-server $BS --id 4
```

In [ ]:
# TODO 6.7 b : arrêter puis désenregistrer
# 1. Arrêtez kafka-4 (`!docker compose --profile scale stop kafka-4`).
# 2. Affichez le quorum (kafka-metadata-quorum.sh ... describe --replication) : kafka-4 y figure encore.
# 3. unregistration : sortie de `!docker exec kafka-tools kafka-cluster.sh unregister --bootstrap-server $BS --id 4`.
# 4. final_brokers : liste triée des identifiants de describe_cluster().
unregistration = ""
final_brokers = []

print(unregistration, final_brokers)

In [ ]:
# Vérification 6.7 b
assert "no longer registered" in unregistration, "À revoir : broker 4 désenregistré"
print("OK  broker 4 désenregistré")
assert final_brokers == [1, 2, 3], "À revoir : brokers 1, 2, 3"
print("OK  brokers 1, 2, 3")
assert not [k for k, (_, r) in placement().items() if 4 in r] and urp() == 0, "À revoir : aucune réplique orpheline, URP à 0"
print("OK  aucune réplique orpheline, URP à 0")

**Résultat attendu** :

- topics sur le broker 4 : `orders`, `orders-history`, `payments` ; le plan ne contient que les brokers 1, 2 et 3 ;
- après la réassignation, le répertoire de `kafka-4` est vide ;
- après l'arrêt, le quorum montre encore le nœud 4 en `Observer`, avec un lag qui ne bouge plus ;
- `Broker 4 is no longer registered.` ; restent les brokers 1, 2 et 3, sans partition sous-répliquée.

## Bilan de l'exercice

In [ ]:
table([
    ("kafka-4 enregistré puis retiré", "%s puis %s" % (sorted(brokers), final_brokers), "6.1, 6.7"),
    ("Répliques sur le broker 4 après 6.4", distribution_74.get(4, "à compléter") if distribution_74 else "à compléter", "6.4"),
    ("Répertoires de kafka-1", ", ".join(log_dirs_1) or "à compléter", "6.5"),
    ("Rolling restart : ordre, messages confirmés", "%s, %s" % (order, load), "6.6"),
    ("Partitions sous-répliquées", urp(), "fin"),
], ["Contrôle", "Valeur", "Étape"])

## 7. Questions de réflexion

1. Pourquoi un broker ajouté ne reçoit-il aucune partition existante ? Que se passe-t-il si l'on oublie de réassigner ?
2. Pourquoi conserver la répartition actuelle affichée par `--generate` ? Comment l'utiliseriez-vous ?
3. Que se passe-t-il si l'on oublie de lancer `--verify` jusqu'au bout après un `--throttle` ?
4. Pourquoi kafka-4 est-il `Observer` et non `Follower` dans le quorum ?
5. Une partition vit dans un seul log dir. Quel impact sur l'équilibre entre les disques ? Que se passe-t-il si le disque `data2` tombe en panne ?
6. Pourquoi redémarrer le contrôleur actif en dernier ?
7. Le producteur n'a reçu aucune erreur. Pourquoi ? Qu'est-ce qui aurait changé avec un topic en RF 2 et `min.insync.replicas=2` ?
8. Pourquoi cordonner le broker **avant** de réassigner ? Pourquoi l'arrêter **avant** de le désenregistrer ?